# overlays

> Popovers, menus, dialogs, drawers, and tooltips

In [ ]:
#| default_exp overlays

All five components float above the page without changing its layout. Native popovers close on Escape or a click outside them (light dismiss). While a modal dialog is open, the browser marks the rest of the page inert. AnswerUI keeps those browser behaviours, applies a shared floating surface, and places popovers beside their opener. Dropdown menus use popovers. Tooltips use CSS anchor positioning instead of the Popover API.

In [ ]:
from fasthtml.common import Div, Span, A, P, Small, H2, Hr, FT, unqid, Dialog as _Dialog
from answerui.registry import hypelt
from answerui.controls import Btn
from answerui.layout import Stack, Cluster
from answerui.demo import example


In [ ]:
from fasthtml.common import *
from answerui.demo import demo
from fastcore.test import *


## Popover

`Popover` renders a floating panel on the raised neutral surface (`--color-surface`). Set `popovertarget` on a button to open it. The browser closes the panel on Escape or a click outside it. While the panel is open, `mountPopover` sets `aria-expanded="true"` on the opener, and `Btn` displays its pressed style.

CSS anchor rules place the panel beside the opener and flip it when it would overflow the viewport. In browsers without anchor positioning, `mountPopover` computes coordinates in JavaScript and closes the panel if the opener leaves the viewport. Pass `side` (`'bottom'`), `align` (`'start'`), and `side_offset` (`6` pixels) to configure initial placement. `Popover` emits `popover`, `popup`, `data-side`, `data-align`, and `data-side-offset` for `mountPopover` with no per-panel JavaScript.

In [ ]:
@hypelt(sel='[popup][popover]')
def Popover(*c, popover='auto', side='bottom', align='start', side_offset=6, **attrs):
    "A native popover positioned beside its opener by the shared controller"
    return Div(*c, popover=popover, popup=True, data_side=side, data_align=align, data_side_offset=side_offset, **attrs)

The shared stylesheet sets the floating surface, viewport size limits, and entry animation. The browser's Popover API handles hiding, light dismiss, and focus restoration.


Set `popovertarget` on any button to the popover's `id` to open it. Put ordinary HTML content inside the panel:

In [ ]:
@example(Popover)
def _popover_example():
    id = unqid()
    return Cluster(Btn('What is a workspace?', variant='secondary', popovertarget=id),
        Popover(P("A workspace holds a team's projects, members, and billing."), id=id, style='max-inline-size: 18rem'))

In [ ]:
Popover.demo(height='9rem')

```html
<div cluster>
<button type="button" btn btn-variant="secondary" popovertarget="_Ve6Wo9qRTwK3VQYryBi0hA">What is a workspace?</button>  <div popover="auto" popup data-side="bottom" data-align="start" data-side-offset="6" id="_Ve6Wo9qRTwK3VQYryBi0hA" style="max-inline-size: 18rem">
    <p>A workspace holds a team's projects, members, and billing.</p>
  </div>
</div>

```

## Dropdown

`Dropdown` returns a `(trigger, panel)` pair containing a button trigger and a native popover of links, buttons, separators, and supplementary content. Pass a label string to create a secondary `Btn` trigger, or pass a configured `Btn` to control its variant and attributes. Give an icon-only trigger an accessible name such as `aria_label`. `Dropdown` always sets `type='button'` on the trigger so it never submits an enclosing form.

Dropdown items are ordinary HTML links and buttons rather than an ARIA `role="menu"` widget. Users move between them with Tab and Shift+Tab, skipping disabled buttons. When the panel opens, `mountPopover` focuses the first enabled visible control inside it. `Dropdown` adds `menu-item` to each direct `a` and `button` child and sets `popovertarget` and `popovertargetaction='hide'` on buttons that do not already have a `popovertarget`. Clicking a link with `href` closes the panel in `mountPopover`. The browser handles Escape and light dismiss. To mark an unavailable item, set `disabled=True` on a button or omit `href` on a link.

In [ ]:
@hypelt(sel='[dropdown][popover]')
def Dropdown(
    trigger,  # Label text or a configured Btn
    *items,   # Buttons, links, separators and supplementary content
    id=None,  # Panel id; generated when omitted
    **attrs,  # Attributes for the panel
):
    "A button and its native popover of ordinary links and actions"
    id = id or unqid()
    if not isinstance(trigger, FT) or trigger.tag != 'button': trigger = Btn(trigger, variant='secondary')
    acts = [c for c in items if isinstance(c, FT) and c.tag in ('a', 'button')]
    for c in acts:
        c(menu_item=True)
        if c.tag == 'button' and 'popovertarget' not in c.attrs: c(popovertarget=id, popovertargetaction='hide')
    return trigger(type='button', popovertarget=id), Div(*items, popover='auto', popup=True, dropdown=True, id=id, **attrs)

The Account example uses a text trigger and starts with the user's name and email aligned with its items. The Options example passes a configured `Btn` with `aria_label`. Tabbing through the items skips both the static header text and the disabled Duplicate button:

In [ ]:
@example(Dropdown)
def _dropdown_example():
    trigger, panel = Dropdown(Btn('Options', aria_label='Project actions'),
        Btn('Rename'), Btn('Duplicate', disabled=True),
        Hr(), Btn('Delete', ui_tone='danger'))
    assert trigger.get('aria-label') == 'Project actions'
    return Cluster(Dropdown('Account', P('Ada Lovelace'), Small('ada@example.com'), Hr(), A('Profile', href='#'), A('Billing', href='#'),
        Hr(), Btn('Sign out')), trigger, panel)

In [ ]:
Dropdown.demo(height='14rem')

```html
<div cluster>
<button type="button" btn btn-variant="secondary" popovertarget="_6KY2jkE1ROS66p7qkdZoKg">Account</button>  <div popover="auto" popup dropdown id="_6KY2jkE1ROS66p7qkdZoKg">
    <p>Ada Lovelace</p>
<small>ada@example.com</small><hr><a href="#" menu-item>Profile</a><a href="#" menu-item>Billing</a><hr><button type="button" btn btn-variant="primary" menu-item popovertarget="_6KY2jkE1ROS66p7qkdZoKg" popovertargetaction="hide">Sign out</button>  </div>
<button type="button" btn btn-variant="primary" aria-label="Project actions" popovertarget="_0KJRJJojQcu_7FwhLSxnxw">Options</button>  <div popover="auto" popup dropdown id="_0KJRJJojQcu_7FwhLSxnxw">
<button type="button" btn btn-variant="primary" menu-item popovertarget="_0KJRJJojQcu_7FwhLSxnxw" popovertargetaction="hide">Rename</button><button type="button" btn btn-variant="primary" disabled menu-item popovertarget="_0KJRJJojQcu_7FwhLSxnxw" popovertargetaction="hide">Duplicate</button><hr><button type="button" btn btn-variant="primary" ui-tone="danger" menu-item popovertarget="_0KJRJJojQcu_7FwhLSxnxw" popovertargetaction="hide">Delete</button>  </div>
</div>

```

## Invoker commands

Set `commandfor` on a button to a dialog's `id` and `command` to `'show-modal'` or `'close'`. In browsers without native Invoker Commands support, the shared click handler in `browser.js` calls `showModal()` or `close()` on the target dialog. Use `popovertarget` for popovers.

## Dialog

`Dialog` renders a native `<dialog>` element with `dialog` and `dialog-side` attributes. Open it as a modal with `command='show-modal'` and `commandfor`. The browser moves focus into the dialog, closes it on Escape, and marks the rest of the page inert. Because the dialog stays in its DOM position while rendered in the top layer, it inherits its font and custom properties from its parent element. Close it with `command='close'` or a `<form method="dialog">`.

The shared stylesheet sets the floating surface (`--color-surface`), rounded corners (`--radius1`), entry animation (`ui-dialog-enter`), and backdrop tint. Add `closedby='any'` so a click on the backdrop closes the dialog. Omit `closedby='any'` on confirmations where the user must click a button.

In [ ]:
@hypelt(sel='dialog[dialog]')
def Dialog(*c, side='center', **attrs):
    "A native modal panel, or a full-height drawer at the start or end edge"
    return _Dialog(*c, dialog=True, dialog_side=side, **attrs)

The htmx listener in `browser.js` calls `showModal()` on a closed `Dialog` after htmx swaps content into a descendant inside it. A button that loads the dialog body with `hx-get` does not need `command='show-modal'`. Because the dialog opens after the swap finishes, users never see stale content. Swapping the `<dialog>` element itself does not open it. A server can send a closed `<dialog>` out of band to close an open one.

This confirmation dialog uses a `Stack` for its body and a `Cluster` for its action buttons. Set `aria_labelledby` to the heading's `id` to give the dialog an accessible name. The browser does not derive a dialog's accessible name from a nested heading:

In [ ]:
@example(Dialog)
def _dialog_example():
    id, title = unqid(), unqid()
    return Cluster(Btn('Delete project', ui_tone='danger', command='show-modal', commandfor=id),
        Dialog(Stack(H2('Delete this project?', id=title), P('Everything in it is removed. This cannot be undone.'),
            Cluster(Btn('Cancel', variant='secondary', command='close', commandfor=id, autofocus=True),
                Btn('Delete', ui_tone='danger', command='close', commandfor=id), cls='gap-2')),
            id=id, aria_labelledby=title))

In [ ]:
Dialog.demo(height='16rem')

```html
<div cluster>
<button type="button" btn btn-variant="primary" ui-tone="danger" command="show-modal" commandfor="_TB2jmKNLTWaZigchjN8yBw">Delete project</button><dialog dialog dialog-side="center" aria-labelledby="_bTcmAUUvS12_Dv2rYZ2P8A" id="_TB2jmKNLTWaZigchjN8yBw">    <div stack>
      <h2 id="_bTcmAUUvS12_Dv2rYZ2P8A">Delete this project?</h2>
      <p>Everything in it is removed. This cannot be undone.</p>
      <div cluster style="--gap: var(--space-2);">
<button type="button" btn btn-variant="secondary" command="close" commandfor="_TB2jmKNLTWaZigchjN8yBw" autofocus>Cancel</button><button type="button" btn btn-variant="primary" ui-tone="danger" command="close" commandfor="_TB2jmKNLTWaZigchjN8yBw">Delete</button>      </div>
    </div>
</dialog></div>

```

## Drawer

Pass `side='end'` or `side='start'` to `Dialog` to place it against the inline edge of the viewport (right and left in left-to-right pages). The drawer uses `--drawer-size` (`24rem` by default) for its width and expands to `100vw` on viewports narrower than `40rem`. Tall content scrolls inside the drawer. A drawer opens and closes like a centred `Dialog`, with the same Escape and page-inertness behaviour. Add `closedby='any'` so a click outside closes the drawer in supporting browsers. A button with `command='close'` closes it in all browsers.

In [ ]:
demo(Cluster(Btn('Filters', variant='secondary', command='show-modal', commandfor='filters'),
    Btn('Navigation', variant='secondary', command='show-modal', commandfor='nav-drawer'),
    Dialog(Stack(H2('Filters', id='filters-title'), P('Narrow the list.'),
        Btn('Done', command='close', commandfor='filters')),
        id='filters', aria_labelledby='filters-title', side='end', closedby='any'),
    Dialog(Stack(H2('Navigation', id='nav-title'), P('Sections of the app.'),
        Btn('Close', variant='secondary', command='close', commandfor='nav-drawer')),
        id='nav-drawer', aria_labelledby='nav-title', side='start', closedby='any')), height='12rem')

```html
<div cluster>
<button type="button" btn btn-variant="secondary" command="show-modal" commandfor="filters">Filters</button><button type="button" btn btn-variant="secondary" command="show-modal" commandfor="nav-drawer">Navigation</button><dialog dialog dialog-side="end" aria-labelledby="filters-title" closedby="any" id="filters">    <div stack>
      <h2 id="filters-title">Filters</h2>
      <p>Narrow the list.</p>
<button type="button" btn btn-variant="primary" command="close" commandfor="filters">Done</button>    </div>
</dialog><dialog dialog dialog-side="start" aria-labelledby="nav-title" closedby="any" id="nav-drawer">    <div stack>
      <h2 id="nav-title">Navigation</h2>
      <p>Sections of the app.</p>
<button type="button" btn btn-variant="secondary" command="close" commandfor="nav-drawer">Close</button>    </div>
</dialog></div>

```

## Tooltip

`Tooltip` pairs a control with a short supplementary description inside an inline `span[tooltip]`. It appends the description's `id` to the control's `aria-describedby` without overwriting existing IDs. Give the control its own accessible name because a tooltip never replaces a label. Use `Popover` for floating panels that contain interactive controls.

The description is real HTML shown by CSS alone. It appears immediately when the pointer is over the control or description, or when the control has keyboard focus (`:focus-visible`). Clicking the control does not show the tooltip. Once hover and keyboard focus end, the tooltip stays visible for `150ms` and then fades out over `--motion`. You can move the pointer onto the description during that `150ms` delay. Moving to another tooltip hides the first one immediately with no fade, matching native tooltips.

`Tooltip` links the control and description with inline `anchor-name` and `position-anchor` styles. With CSS anchor positioning, the browser centres the description above the control with a `--tooltip-gap` (`6px` default) gap and flips it below when there is no room above. In browsers without anchor positioning, the description is placed above the control and aligned to its left edge. Tooltips render outside the top layer rather than using the Popover API. An ancestor with `transform` or `filter` becomes the containing block and can clip the tooltip.

In [ ]:
@hypelt(sel='[tooltip]')
def Tooltip(
    trigger, # A control with its own accessible name
    text,    # Supplementary, non-interactive content
    id=None, # Description id; generated when omitted
    **attrs, # Attributes for the description
):
    "A control and supplementary description shown by CSS on hover or keyboard focus"
    id = id or unqid()
    describedby = ' '.join(filter(None, [trigger.get('aria-describedby'), id]))
    trigger(aria_describedby=describedby, style=' '.join(filter(None, [trigger.get('style'), f'anchor-name: --{id};'])))
    tip = Div(text, role='tooltip', id=id, **attrs)(style=' '.join(filter(None, [attrs.get('style'), f'position-anchor: --{id};'])))
    return Span(trigger, tip, tooltip=True)


In [ ]:
@example(Tooltip)
def _tooltip_example():
    return Cluster(Tooltip(Btn('Archive', variant='secondary'), 'Move to the archive'),
        Tooltip(Btn('Share', variant='secondary'), 'Copy a link'),
        Tooltip(Btn('Delete', ui_tone='danger'), 'This cannot be undone'))

In [ ]:
Tooltip.demo(height='7rem')

```html
<div cluster>
<span tooltip><button type="button" btn btn-variant="secondary" aria-describedby="_r1AOiS-ASUWwlpjHZs_WpA" style="anchor-name: --_r1AOiS-ASUWwlpjHZs_WpA;">Archive</button>    <div role="tooltip" id="_r1AOiS-ASUWwlpjHZs_WpA" style="position-anchor: --_r1AOiS-ASUWwlpjHZs_WpA;">Move to the archive</div>
</span><span tooltip><button type="button" btn btn-variant="secondary" aria-describedby="_laieiGlzROKBomXnGiwreg" style="anchor-name: --_laieiGlzROKBomXnGiwreg;">Share</button>    <div role="tooltip" id="_laieiGlzROKBomXnGiwreg" style="position-anchor: --_laieiGlzROKBomXnGiwreg;">Copy a link</div>
</span><span tooltip><button type="button" btn btn-variant="primary" ui-tone="danger" aria-describedby="_FmmJihkrTVSIyVenoChpQg" style="anchor-name: --_FmmJihkrTVSIyVenoChpQg;">Delete</button>    <div role="tooltip" id="_FmmJihkrTVSIyVenoChpQg" style="position-anchor: --_FmmJihkrTVSIyVenoChpQg;">This cannot be undone</div>
</span></div>

```